# 教程 04：AI for PDE

**学习目标** ⭐
- 了解 AI for PDE 的核心理念
- 学习 PINN / FNO / DeepONet / MeshGraphNet 的基本原理
- 掌握本仓库 AI4PDE 算子的使用方式

---

## 什么是 AI for PDE?

偏微分方程 (PDE) 是描述物理规律的核心数学工具（流体力学用 Navier-Stokes，传热用热传导方程，结构力学用弹性方程）。
传统 PDE 求解器（FEM、FDM）计算量大、网格依赖强。AI for PDE 用神经网络替代或加速求解器。

本仓库提供了 4 种主流方法：

| 方法 | 原理 | 适用场景 |
|------|------|----------|
| **PINN** | 物理信息神经网络 (Physics-Informed NN) | 正问题、逆问题、数据稀少 |
| **FNO** | 傅里叶神经算子 (Fourier Neural Operator) | 参数量大、分辨率不变、数据驱动 |
| **DeepONet** | 深度算子网络 | 不同边界/初始条件泛化 |
| **MeshGraphNet** | 图神经网络网格求解 | 不规则网格、复杂几何 |

In [ ]:
import os

repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
ai4pde = os.path.join(repo_root, 'simulation', 'AI4PDE')

print("AI4PDE 算子列表:")
for op in sorted(os.listdir(ai4pde)):
    op_path = os.path.join(ai4pde, op)
    if os.path.isdir(op_path) and op != '__pycache__':
        kernel_files = [f for f in os.listdir(os.path.join(op_path, 'op_kernel'))
                        if f.endswith(('.cpp', '.h'))] if os.path.isdir(os.path.join(op_path, 'op_kernel')) else []
        test_files = [f for f in os.listdir(os.path.join(op_path, 'tests'))
                      if f.endswith('.py')] if os.path.isdir(os.path.join(op_path, 'tests')) else []
        print(f"  📂 {op}")
        if kernel_files:
            print(f"     Kernel: {', '.join(kernel_files)}")
        if test_files:
            print(f"     测试: {', '.join(test_files)}")

## PINN 原理

物理信息神经网络 (PINN) 将 PDE 的残差直接作为损失函数的一部分，迫使网络输出满足物理方程。

以一维 Poisson 方程为例：

$$-\frac{d^2 u}{dx^2} = f(x), \quad x \in [0, 1]$$

$$u(0) = u(1) = 0$$

PINN 的损失函数：
$$L = \frac{1}{N_f} \sum \left| -\frac{d^2 u_\theta}{dx^2} - f(x) \right|^2 + \frac{1}{N_b} \sum |u_\theta - g|^2$$

其中 $u_\theta$ 是神经网络，第一项强制满足方程，第二项强制满足边界条件。

In [ ]:
import torch
import torch.nn as nn

class PINN(nn.Module):
    """简单 PINN 网络：一维 Poisson 方程"""
    def __init__(self, n_hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, n_hidden), nn.Tanh(),
            nn.Linear(n_hidden, n_hidden), nn.Tanh(),
            nn.Linear(n_hidden, n_hidden), nn.Tanh(),
            nn.Linear(n_hidden, 1),
        )

    def forward(self, x):
        return self.net(x)

def pinn_loss(model, x_f, x_b):
    """物理信息损失"""
    x_f.requires_grad_(True)
    u = model(x_f)
    du = torch.autograd.grad(u, x_f, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    d2u = torch.autograd.grad(du, x_f, grad_outputs=torch.ones_like(du), create_graph=True)[0]
    f = -torch.sin(torch.pi * x_f) * torch.pi**2  # 精确解为 sin(πx)
    pde_loss = torch.mean((d2u + f) ** 2)
    u_b = model(x_b)
    bc_loss = torch.mean(u_b ** 2)
    return pde_loss + bc_loss

model = PINN()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

x_f = torch.rand(100, 1)
x_b = torch.tensor([[0.0], [1.0]])

print("训练 PINN...")
for epoch in range(500):
    optimizer.zero_grad()
    loss = pinn_loss(model, x_f, x_b)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"  Epoch {epoch}: loss = {loss.item():.6f}")
print("训练完成")

In [ ]:
try:
    import matplotlib.pyplot as plt

    with torch.no_grad():
        x_test = torch.linspace(0, 1, 100).unsqueeze(-1)
        u_pred = model(x_test).squeeze()
        u_exact = torch.sin(torch.pi * x_test.squeeze())

    plt.figure(figsize=(8, 4))
    plt.plot(x_test.numpy(), u_exact.numpy(), 'b-', label='精确解 sin(πx)', linewidth=2)
    plt.plot(x_test.numpy(), u_pred.numpy(), 'r--', label='PINN 预测', linewidth=2)
    plt.xlabel('x')
    plt.ylabel('u(x)')
    plt.title('PINN 求解一维 Poisson 方程')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    error = torch.abs(u_pred - u_exact).mean().item()
    print(f"平均绝对误差: {error:.4e}")
except ImportError:
    print("需要 matplotlib")

## FNO (Fourier Neural Operator)

FNO 的核心思想：在傅里叶空间进行卷积操作。

$$v_{t+1}(x) = \sigma\left(W v_t(x) + \mathcal{F}^{-1}\left(R_\phi \cdot \mathcal{F}(v_t)\right)(x)\right)$$

其中 $\mathcal{F}$ 是傅里叶变换，$R_\phi$ 是学习的权重滤波器。

本仓库的 FNO 算子在 `simulation/AI4PDE/fno/` 下，包含完整的 Ascend C 实现。

## 总结 📋

- AI for PDE 是 AI + 科学计算的前沿方向
- PINN 将 PDE 残差作为损失函数，适合正/逆问题
- FNO 在傅里叶空间学习算子映射，分辨率不变
- DeepONet 学习函数到函数的映射
- MeshGraphNet 在图网格上求解 PDE
- 本仓库提供了 4 种方法的 Ascend C 算子实现

## 思考练习 ✏️

1. 修改 PINN 的网络深度和宽度，观察精度变化
2. 尝试求解二维 Poisson 方程 $-(u_{xx} + u_{yy}) = f(x,y)$
3. 阅读本仓库 FNO 算子的 Kernel 代码，理解傅里叶层如何在 NPU 上实现